# ARC-v0.36.3 — NQ-GTE Frozen Downstream QA Consequence Audit
## Gold-Alignment Fix Before Any Answer Outcomes

**Scientific role:** implementation/data-alignment repair of ARC-v0.36, performed **before any answer-generation outcome was produced**.

ARC-v0.36 correctly stopped at its gold-alignment guard: only 326/500 frozen BEIR NQ queries matched `NQ-open.dev.jsonl` by normalized text. No answer-generation cells were executed. The failure exposed a source-split mismatch, not a model or retrieval outcome.

ARC-v0.36.3 changes **only the authoritative QA gold source and its alignment audit**. It does **not** change:

- the frozen 500-query membership;
- GTE-small embeddings;
- PQ32@64 representation intervention;
- SQ8@2 search-effort intervention;
- SQ8@64 shared higher-fidelity branch;
- anchored centroid feedback;
- H=4;
- the frozen eight-policy subset;
- Qwen2.5-3B-Instruct answerer;
- top-5 evidence serialization;
- token-F1 primary metric;
- difference-in-differences primary estimand;
- query-level 10,000-replicate paired bootstrap.

## Why the gold source changes

BEIR Natural Questions follows the DPR open-domain retrieval lineage. DPR explicitly distributes question-answer CSV files for NQ train/dev/test and documents the `data.retriever.qas.*` resources as the evaluation QA source. ARC-v0.36.3 therefore uses DPR's `nq-test.qa.csv`, not Google NQ-Open's `NQ-open.dev.jsonl`, and requires **100% normalized exact-question coverage for the already frozen 500 queries** before any answer generation may begin.

Authoritative DPR QA URL used here:

`https://dl.fbaipublicfiles.com/dpr/data/retriever/nq-test.qa.csv`

## Primary estimand (unchanged)

For query q,

`D0(q) = F1_search_0(q) - F1_rep_0(q)`

and, after averaging terminal answer F1 over the eight frozen policies,

`DH(q) = mean_p [F1_search_H(q,p) - F1_rep_H(q,p)]`.

The frozen primary remains

`Delta_QA = mean_q [DH(q) - D0(q)]`.

A positive CI means feedback adds a downstream answer-quality advantage for the search-effort mechanism relative to representation approximation beyond the one-shot answer difference. A negative CI is the reverse. A CI crossing zero is unresolved.

## Evidence-status guardrail

This is not a new pristine retrieval confirmation. It is a **pre-answer-outcome data-source repair**. The original ARC-v0.36 failure and its 326/500 coverage should be retained in notes/provenance rather than erased.

In [ ]:
# Cell 1 — Install pinned / bounded dependencies
import sys, subprocess

pkgs = [
    "faiss-cpu==1.12.0",
    "sentence-transformers>=5.0,<6",
    "transformers>=4.55,<5",
    "accelerate>=1.5",
    "huggingface_hub>=0.34",
    "pyarrow",
    "pandas",
    "numpy",
    "tqdm",
    "requests",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencies installed.")

In [ ]:
# Cell 2 — Imports, constants, and Drive paths
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, os, random, re, shutil, string, unicodedata, zipfile

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from tqdm.auto import tqdm
from huggingface_hub import model_info
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

SEED = 20260836
DIM = 384
H = 4
TOP_RETRIEVE = 100
EVIDENCE_K = 5
MAX_PASSAGE_CHARS = 850
BOOTSTRAP_REPS = 10_000

REP_NPROBE = 64
SEARCH_LOW_NPROBE = 2
SEARCH_HIGH_NPROBE = 64

ANSWER_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ANSWER_MAX_NEW_TOKENS = 32
ANSWER_BATCH_SIZE = 8
N_MAIN_EXPECTED = 500

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
RAG_ROOT = DRIVE_ROOT / "rag-pq-checkpoints"
ARC_ROOT = RAG_ROOT / "arc-v0"
LARGE_ROOT = RAG_ROOT / "arc-v027-nq-gte-large-cache"
V031_ROOT = ARC_ROOT / "generative-retrieval-agent-transfer-v031"
V032_ROOT = ARC_ROOT / "nq-gte-exhaustive-reference-v032"
V036_ROOT = ARC_ROOT / "nq-gte-downstream-qa-consequence-v036"
V036_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V036_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PQ_PATH = LARGE_ROOT / "nq-gte-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"
QUERY_IDS_PATH = LARGE_ROOT / "nq_query_ids.txt"
QUERY_EMB_PATH = LARGE_ROOT / "nq_query_embeddings.float32.npy"
CORPUS_MANIFEST_PATH = LARGE_ROOT / "nq_corpus_block_manifest.json"
QREL_ROW_MAP_PATH = LARGE_ROOT / "nq_qrel_doc_rows.csv"
V031_PROTOCOL_PATH = V031_ROOT / "V031_FROZEN_PROTOCOL.json"

required = [
    PQ_PATH, SQ_PATH, QUERY_IDS_PATH, QUERY_EMB_PATH,
    CORPUS_MANIFEST_PATH, QREL_ROW_MAP_PATH, V031_PROTOCOL_PATH,
]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, "Missing source artifacts:\n" + "\n".join(missing)

print("OUT:", OUT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Cell 3 — Provenance helpers and frozen 500-query membership
def sha256_file(path, chunk=16*1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256("\n".join(sorted(map(str, ids))).encode("utf-8")).hexdigest()

def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

manifest = json.loads(CORPUS_MANIFEST_PATH.read_text())
assert manifest["status"] == "COMPLETE"
assert manifest["encoder"] == "thenlper/gte-small"
assert int(manifest["dimension"]) == DIM
assert int(manifest["corpus_rows"]) == 2_681_468

ALL_QUERY_IDS = QUERY_IDS_PATH.read_text(encoding="utf-8").splitlines()
query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode="r")
assert query_embeddings.shape == (len(ALL_QUERY_IDS), DIM)
qid_to_pos = {q:i for i,q in enumerate(ALL_QUERY_IDS)}

v031 = json.loads(V031_PROTOCOL_PATH.read_text())
MAIN_IDS = list(map(str, v031["main_ids"]))
assert len(MAIN_IDS) == N_MAIN_EXPECTED and len(set(MAIN_IDS)) == N_MAIN_EXPECTED
assert set(MAIN_IDS).issubset(set(ALL_QUERY_IDS))

POLICIES = []
for alpha in [0.1, 0.3, 0.5, 0.7]:
    POLICIES.append({"name":f"mean-k20-a{alpha}", "family":"mean", "k":20, "tau":None, "alpha":alpha})
    POLICIES.append({"name":f"softmax-k20-t0.1-a{alpha}", "family":"softmax", "k":20, "tau":0.1, "alpha":alpha})
assert len(POLICIES) == 8

print("Main queries:", len(MAIN_IDS))
print("Main membership SHA:", membership_sha(MAIN_IDS))

In [ ]:
# Cell 4 — Load BEIR NQ text and authoritative DPR NQ-test QA gold
# ARC-v0.36.3 parser repair:
# DPR nq-test.qa.csv is a TAB-delimited file that also uses CSV-style quoting.
# Some answer lists are wrapped in double quotes and escape embedded double quotes
# by doubling them. Therefore, parse with csv.reader(delimiter="\t", quotechar='"',
# doublequote=True) instead of raw string splitting.

import ast
import csv

LOCAL_RAW = Path("/content/arc-v0363-nq")
LOCAL_RAW.mkdir(parents=True, exist_ok=True)

# BEIR NQ retrieval dataset
NQ_ZIP = LOCAL_RAW / "nq.zip"
NQ_DIR = LOCAL_RAW / "nq"
BEIR_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip"

if not (NQ_DIR / "corpus.jsonl").is_file():
    if not NQ_ZIP.is_file():
        with requests.get(BEIR_URL, stream=True, timeout=180) as r:
            r.raise_for_status()
            with open(NQ_ZIP, "wb") as f:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
    with zipfile.ZipFile(NQ_ZIP) as z:
        z.extractall(LOCAL_RAW)

CORPUS_JSONL = NQ_DIR / "corpus.jsonl"
QUERIES_JSONL = NQ_DIR / "queries.jsonl"
QRELS_TSV = NQ_DIR / "qrels" / "test.tsv"

QUERY_TEXT = {}
with open(QUERIES_JSONL, encoding="utf-8") as f:
    for line in f:
        o = json.loads(line)
        QUERY_TEXT[str(o["_id"])] = str(o.get("text", ""))

assert all(q in QUERY_TEXT for q in MAIN_IDS)

# DPR's NQ test QA evaluation source.
DPR_NQ_TEST_QA_URL = "https://dl.fbaipublicfiles.com/dpr/data/retriever/nq-test.qa.csv"
DPR_QA_PATH = LOCAL_RAW / "nq-test.qa.csv"

if not DPR_QA_PATH.is_file():
    with requests.get(DPR_NQ_TEST_QA_URL, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(DPR_QA_PATH, "wb") as f:
            for chunk in r.iter_content(4 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

def norm_question(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_answers_field(s):
    s = str(s).strip()
    # After csv.reader removes outer CSV quoting / doubled quote escapes,
    # the field should be a Python-list-like or JSON-list-like expression.
    for parser in (ast.literal_eval, json.loads):
        try:
            v = parser(s)
            if isinstance(v, (list, tuple)):
                vals = [str(x) for x in v]
                if vals:
                    return vals
        except Exception:
            pass
    raise ValueError(f"Could not parse DPR answer field: {s[:300]!r}")

gold_by_q = {}
duplicates = defaultdict(list)
bad_rows = []

with open(DPR_QA_PATH, encoding="utf-8", newline="") as f:
    reader = csv.reader(
        f,
        delimiter="\t",
        quotechar='"',
        doublequote=True,
        quoting=csv.QUOTE_MINIMAL,
        strict=False,
    )
    for line_no, row in enumerate(reader, start=1):
        if not row or not any(str(x).strip() for x in row):
            continue

        if len(row) != 2:
            bad_rows.append((line_no, row[:5]))
            continue

        question, answers_raw = row

        # Optional defensive header handling.
        if line_no == 1 and norm_question(question) in {"question", "query"}:
            continue

        try:
            answers = parse_answers_field(answers_raw)
        except Exception as e:
            bad_rows.append((line_no, row, repr(e)))
            continue

        duplicates[norm_question(question)].append(answers)

assert not bad_rows, f"Malformed DPR QA rows: {bad_rows[:5]}"
assert duplicates, "No DPR QA rows parsed."

for k, vals in duplicates.items():
    merged = []
    for xs in vals:
        for x in xs:
            if x not in merged:
                merged.append(x)
    gold_by_q[k] = merged

GOLD = {}
unmatched = []
for qid in MAIN_IDS:
    k = norm_question(QUERY_TEXT[qid])
    if k not in gold_by_q:
        unmatched.append((qid, QUERY_TEXT[qid]))
    else:
        GOLD[qid] = gold_by_q[k]

coverage = len(GOLD) / len(MAIN_IDS)

alignment_report = {
    "study_id": "ARC-v0.36.3",
    "repair": (
        "DPR quoted-TSV parser fix only: csv.reader with delimiter TAB, "
        "quotechar double-quote, doublequote=True"
    ),
    "gold_source": DPR_NQ_TEST_QA_URL,
    "dpr_qa_sha256": sha256_file(DPR_QA_PATH),
    "n_dpr_questions": len(gold_by_q),
    "n_frozen_queries": len(MAIN_IDS),
    "n_matched": len(GOLD),
    "coverage": coverage,
    "unmatched": unmatched,
    "original_v036_failed_source": "Google NQ-open NQ-open.dev.jsonl",
    "original_v036_observed_coverage": "326/500 = 0.652",
    "v0361_failure": "delimiter sniffing treated literal-tab rows as one column",
    "v0362_failure": "raw first-tab split left CSV outer quoting / doubled-quote escapes in answer field",
    "answer_outcomes_before_fix": False,
}
(OUT / "v0363_gold_alignment_report.json").write_text(
    json.dumps(alignment_report, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("DPR QA questions parsed:", len(gold_by_q))
print("Gold coverage:", len(GOLD), "/", len(MAIN_IDS), "=", coverage)
if unmatched:
    print("Unmatched examples:", unmatched[:20])

# Hard gate: do not shrink or alter the frozen 500-query membership.
assert len(GOLD) == len(MAIN_IDS) == N_MAIN_EXPECTED, (
    "DPR NQ-test QA source does not cover the full frozen 500-query membership. "
    "STOP. Do not drop unmatched queries and do not generate answer outcomes."
)

empty = [
    qid for qid, ans in GOLD.items()
    if not ans or not any(str(x).strip() for x in ans)
]
assert not empty, f"Frozen queries with empty DPR gold answers: {empty[:20]}"

print("DPR NQ-TEST GOLD ALIGNMENT — 100% PASS")

In [ ]:
# Cell 5 — Build corpus row offsets and verify row semantics against persisted qrel mapping
OFFSETS_PATH = LOCAL_RAW / "corpus_offsets.npy"
DOCIDS_PATH = LOCAL_RAW / "corpus_docids.txt"

if not OFFSETS_PATH.is_file() or not DOCIDS_PATH.is_file():
    offsets, ids = [], []
    with open(CORPUS_JSONL, "rb") as f:
        while True:
            p = f.tell()
            line = f.readline()
            if not line:
                break
            o = json.loads(line)
            offsets.append(p)
            ids.append(str(o["_id"]))
    np.save(OFFSETS_PATH, np.asarray(offsets, dtype=np.int64))
    DOCIDS_PATH.write_text("\n".join(ids), encoding="utf-8")

OFFSETS = np.load(OFFSETS_PATH, mmap_mode="r")
ROW_DOCIDS = DOCIDS_PATH.read_text(encoding="utf-8").splitlines()
assert len(OFFSETS) == len(ROW_DOCIDS) == int(manifest["corpus_rows"])

rel_map = pd.read_csv(QREL_ROW_MAP_PATH)
rel_map["doc_id"] = rel_map["doc_id"].astype(str)
sample = rel_map.sample(n=min(1000, len(rel_map)), random_state=SEED)
for _, r in sample.iterrows():
    row = int(r["corpus_row"])
    assert ROW_DOCIDS[row] == str(r["doc_id"]), (row, ROW_DOCIDS[row], r["doc_id"])

corpus_fh = open(CORPUS_JSONL, "rb")

def corpus_row(row):
    row = int(row)
    corpus_fh.seek(int(OFFSETS[row]))
    return json.loads(corpus_fh.readline())

def evidence_for_rows(rows, k=EVIDENCE_K):
    out = []
    for r in rows[:k]:
        r = int(r)
        if r < 0:
            continue
        o = corpus_row(r)
        title = str(o.get("title","") or "").strip()
        text = str(o.get("text","") or "").replace("\n"," ").strip()
        out.append({
            "row": r,
            "doc_id": str(o["_id"]),
            "title": title,
            "text": text[:MAX_PASSAGE_CHARS],
        })
    return out

print("ROW SEMANTICS — PASS")

In [ ]:
# Cell 6 — Localize persisted corpus embedding blocks and load vector store
LOCAL_BLOCK_ROOT = Path("/content/arc-v036-nq-corpus-blocks")
LOCAL_BLOCK_ROOT.mkdir(parents=True, exist_ok=True)

BLOCK_PATHS = []
for rec in tqdm(manifest["blocks"], desc="Corpus blocks"):
    src = Path(rec["path"])
    assert src.is_file(), src
    dst = LOCAL_BLOCK_ROOT / src.name
    if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    BLOCK_PATHS.append(dst)

class BlockEmbeddingStore:
    def __init__(self, paths, dim=384, block_rows=100_000):
        self.paths = list(paths)
        self.dim = dim
        self.block_rows = block_rows
        self.cache = {}
    def _block(self, b):
        b = int(b)
        if b not in self.cache:
            self.cache[b] = np.load(self.paths[b], mmap_mode="r")
        return self.cache[b]
    def get(self, ids):
        ids = np.asarray(ids, dtype=np.int64)
        shape = ids.shape
        flat = ids.reshape(-1)
        if np.any(flat < 0) or np.any(flat >= int(manifest["corpus_rows"])):
            raise ValueError("Invalid corpus row id.")
        out = np.empty((len(flat), self.dim), dtype=np.float32)
        blocks = flat // self.block_rows
        for b in np.unique(blocks):
            mask = blocks == b
            local = flat[mask] - int(b)*self.block_rows
            out[mask] = np.asarray(self._block(int(b))[local], dtype=np.float32)
        return out.reshape(*shape, self.dim)

store = BlockEmbeddingStore(BLOCK_PATHS, DIM)
print("Embedding store ready.")

In [ ]:
# Cell 7 — Load frozen ANN indexes and retrieval / feedback helpers
pq_cpu = faiss.read_index(str(PQ_PATH))
sq_cpu = faiss.read_index(str(SQ_PATH))
assert pq_cpu.ntotal == sq_cpu.ntotal == int(manifest["corpus_rows"])
assert pq_cpu.d == sq_cpu.d == DIM
faiss.omp_set_num_threads(max(1, os.cpu_count() or 1))

def set_nprobe(index, nprobe):
    try:
        index.nprobe = int(nprobe)
    except Exception:
        ps = faiss.ParameterSpace()
        ps.set_index_parameter(index, "nprobe", int(nprobe))

def batched_search(index, q, nprobe, k=TOP_RETRIEVE, batch=256):
    set_nprobe(index, nprobe)
    q = np.ascontiguousarray(q, dtype=np.float32)
    Ds, Is = [], []
    for s in range(0, len(q), batch):
        D, I = index.search(q[s:s+batch], int(k))
        Ds.append(np.asarray(D)); Is.append(np.asarray(I))
    return np.vstack(Ds), np.vstack(Is)

def feedback(ids, scores, policy):
    k = int(policy["k"])
    V = store.get(ids[:, :k])
    if policy["family"] == "mean":
        F = V.mean(axis=1)
    else:
        z = scores[:, :k].astype(np.float64) / float(policy["tau"])
        z -= z.max(axis=1, keepdims=True)
        w = np.exp(z)
        w /= w.sum(axis=1, keepdims=True)
        F = np.einsum("nk,nkd->nd", w.astype(np.float32), V)
    return normalize_rows(F)

def anchored_update(q0, F, alpha):
    return normalize_rows((1.0-float(alpha))*q0 + float(alpha)*F)

MAIN_POS = np.asarray([qid_to_pos[q] for q in MAIN_IDS], dtype=np.int64)
Q0 = normalize_rows(np.asarray(query_embeddings[MAIN_POS], dtype=np.float32))
print("Indexes and helpers ready.")

In [ ]:
# Cell 8 — Freeze ARC-v0.36.3 runtime protocol BEFORE any answer generation
MODEL_REVISION = str(model_info(ANSWER_MODEL).sha)

protocol = {
    "study_id": "ARC-v0.36.3",
    "title": "NQ-GTE Frozen Downstream QA Consequence Audit — Gold-Alignment Fixed",
    "scientific_role": (
        "pre-answer-outcome data-alignment repair of ARC-v0.36; "
        "source retrieval outcomes pre-exist, answer outcomes do not"
    ),
    "status": "FROZEN_BEFORE_V0361_ANSWER_OUTCOMES",
    "parser_fix": {
        "v0361_failure": "csv.Sniffer selected wrong delimiter because answer fields contain commas",
        "v0362_failure": "raw first-TAB split left CSV outer quoting and doubled-quote escapes",
        "v0363_change": "csv.reader(delimiter=TAB, quotechar=double-quote, doublequote=True)",
        "answer_outcomes_generated_before_parser_fix": False,
        "scientific_choices_changed": False,
    },
    "data_alignment_fix": {
        "original_v036_gold_source": "Google NQ-open NQ-open.dev.jsonl",
        "original_v036_coverage": "326/500",
        "failure_point": "gold-alignment assertion before answerer load/generation",
        "answer_outcomes_generated_before_fix": False,
        "new_gold_source": DPR_NQ_TEST_QA_URL,
        "new_gold_sha256": sha256_file(DPR_QA_PATH),
        "required_coverage": "500/500 exact normalized-question matches",
        "scientific_choices_changed": False,
    },
    "dataset": "BEIR Natural Questions",
    "gold_answers": "DPR data.retriever.qas.nq-test / nq-test.qa.csv",
    "encoder": "thenlper/gte-small",
    "source_membership": "ARC-v0.31/0.32 frozen 500-query main subset",
    "n_queries": len(MAIN_IDS),
    "main_ids_sha256": membership_sha(MAIN_IDS),
    "H": H,
    "operator": "anchored centroid",
    "policies": POLICIES,
    "retrieval": {
        "representation_low": "IVF-PQ32@nprobe64",
        "search_effort_low": "IVF-SQ8@nprobe2",
        "shared_high": "IVF-SQ8@nprobe64",
        "top_retrieve": TOP_RETRIEVE,
    },
    "answerer": {
        "model": ANSWER_MODEL,
        "revision": MODEL_REVISION,
        "do_sample": False,
        "max_new_tokens": ANSWER_MAX_NEW_TOKENS,
        "evidence_k": EVIDENCE_K,
        "max_passage_chars": MAX_PASSAGE_CHARS,
        "scores_exposed": False,
    },
    "primary": {
        "name": "feedback_added_F1_mechanism_contrast",
        "D0": "F1_search_low_t0 - F1_rep_low_t0",
        "DH": "mean_policy(F1_search_low_H - F1_rep_low_H)",
        "estimand": "mean_query(DH - D0)",
        "classification": {
            "positive": "95% paired-query bootstrap CI strictly above 0",
            "reversed": "95% paired-query bootstrap CI strictly below 0",
            "unresolved": "95% CI includes 0",
        },
    },
    "secondary": [
        "terminal F1 mechanism contrast DH",
        "one-shot F1 mechanism contrast D0",
        "EM analogue of primary",
        "mean gold F1/EM by high, representation-low, and search-low branches",
        "high-minus-low gold F1 for each mechanism",
        "answer-string disagreement with shared high branch",
        "fraction of queries with positive / negative / tied feedback-added contrast",
    ],
    "statistics": {
        "independent_unit": "query",
        "within_query_policy_aggregation": "mean over frozen 8-policy subset",
        "bootstrap_reps": BOOTSTRAP_REPS,
        "seed": SEED,
    },
    "retention_rule": (
        "Retain positive, null, or reversed answer outcomes. "
        "No model/prompt/evidence depth/query/policy/horizon/metric retuning after this freeze."
    ),
    "source_hashes": {
        "pq_index": sha256_file(PQ_PATH),
        "sq_index": sha256_file(SQ_PATH),
        "query_embeddings": sha256_file(QUERY_EMB_PATH),
        "corpus_manifest": sha256_file(CORPUS_MANIFEST_PATH),
        "v031_protocol": sha256_file(V031_PROTOCOL_PATH),
        "dpr_nq_test_qa": sha256_file(DPR_QA_PATH),
    },
}

PROTOCOL_PATH = OUT / "V0363_FROZEN_PROTOCOL.json"
PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2, sort_keys=True), encoding="utf-8")
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)
(OUT / "V0363_PROTOCOL_SHA256.txt").write_text(PROTOCOL_SHA + "\n", encoding="utf-8")

print("Answer model:", ANSWER_MODEL)
print("Pinned revision:", MODEL_REVISION)
print("DPR QA SHA:", sha256_file(DPR_QA_PATH))
print("Protocol SHA:", PROTOCOL_SHA)
print("V0.36.3 PROTOCOL FREEZE — PASS")
print("\nDO NOT MODIFY SCIENTIFIC CHOICES AFTER THIS CELL.")

## Stop point for Git provenance

After the **100% DPR gold-alignment PASS** and runtime protocol freeze above, but **before running the answerer cell**, save/download this notebook and commit the repair.

Recommended commit message:

`Fix ARC v0.36 gold alignment before answer outcomes`

This is a data-source alignment repair, not outcome-driven retuning. Keep the original v0.36 notebook and its failed 326/500 alignment attempt in Git history.

In [ ]:
# Cell 9 — Deterministic answer prompt, NQ EM/F1 metrics, and answerer load
SYSTEM_PROMPT = (
    "You answer open-domain factual questions from retrieved evidence. "
    "Return only the shortest answer span that answers the question. "
    "Do not explain, cite passages, or add extra text. "
    "If the evidence is insufficient, return your best concise answer."
)

def render_answer_prompt(question, evidence):
    parts = ["QUESTION:\n" + question.strip(), "RETRIEVED PASSAGES:"]
    for rank, item in enumerate(evidence, 1):
        title = item["title"]
        text = item["text"]
        parts.append(f"[{rank}] {title}\n{text}")
    parts.append("ANSWER:")
    return "\n\n".join(parts)

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(unicodedata.normalize("NFKC", str(s)).lower())))

def exact_match(pred, golds):
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in golds))

def token_f1_single(pred, gold):
    p = normalize_answer(pred).split()
    g = normalize_answer(gold).split()
    if len(p) == 0 or len(g) == 0:
        return float(p == g)
    common = {}
    for tok in p:
        common[tok] = min(p.count(tok), g.count(tok))
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(p)
    recall = num_same / len(g)
    return 2 * precision * recall / (precision + recall)

def token_f1(pred, golds):
    return max(token_f1_single(pred, g) for g in golds)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(ANSWER_MODEL, revision=MODEL_REVISION)
answer_model = AutoModelForCausalLM.from_pretrained(
    ANSWER_MODEL,
    revision=MODEL_REVISION,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    answer_model = answer_model.to("cpu")
answer_model.eval()

CACHE_PATH = OUT / "v036_answer_cache.jsonl"
answer_cache = {}
if CACHE_PATH.is_file():
    for line in CACHE_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            o = json.loads(line)
            answer_cache[o["key"]] = o["answer"]

def answer_cache_key(question, evidence):
    payload = {
        "model": ANSWER_MODEL,
        "revision": MODEL_REVISION,
        "system": SYSTEM_PROMPT,
        "question": question,
        "evidence": evidence,
        "do_sample": False,
        "max_new_tokens": ANSWER_MAX_NEW_TOKENS,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def clean_answer(s):
    s = re.sub(r"^```(?:text)?\s*", "", s.strip(), flags=re.I)
    s = re.sub(r"\s*```$", "", s)
    s = re.sub(r"^(answer)\s*:\s*", "", s, flags=re.I)
    return " ".join(s.strip().strip('"').strip("'").split())[:256]

@torch.inference_mode()
def answer_batch(items):
    # items: list[(qid, question, evidence)]
    outputs = [None] * len(items)
    missing = []
    missing_pos = []
    for i, (qid, question, evidence) in enumerate(items):
        key = answer_cache_key(question, evidence)
        if key in answer_cache:
            outputs[i] = answer_cache[key]
        else:
            missing.append((key, qid, question, evidence))
            missing_pos.append(i)

    for s in range(0, len(missing), ANSWER_BATCH_SIZE):
        batch = missing[s:s+ANSWER_BATCH_SIZE]
        prompts = []
        for key, qid, question, evidence in batch:
            user = render_answer_prompt(question, evidence)
            messages = [
                {"role":"system","content":SYSTEM_PROMPT},
                {"role":"user","content":user},
            ]
            prompts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            ))
        toks = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=4096
        )
        device = next(answer_model.parameters()).device
        toks = {k:v.to(device) for k,v in toks.items()}
        generated = answer_model.generate(
            **toks,
            do_sample=False,
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
        )
        input_len = toks["input_ids"].shape[1]
        texts = tokenizer.batch_decode(generated[:, input_len:], skip_special_tokens=True)
        for (key, qid, question, evidence), txt in zip(batch, texts):
            ans = clean_answer(txt)
            answer_cache[key] = ans
            with CACHE_PATH.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"key":key,"answer":ans}, ensure_ascii=False) + "\n")

    for i, (qid, question, evidence) in enumerate(items):
        outputs[i] = answer_cache[answer_cache_key(question, evidence)]
    return outputs

print("ANSWERER LOAD — PASS")

In [ ]:
# Cell 10 — Determinism smoke test (engineering-only, no main outcome inference)
smoke_qids = MAIN_IDS[:4]
_, smoke_rows = batched_search(sq_cpu, Q0[:4], SEARCH_HIGH_NPROBE)

for qid, rows in zip(smoke_qids, smoke_rows):
    ev = evidence_for_rows(rows)
    q = QUERY_TEXT[qid]
    # Bypass persistent result semantics by directly invoking identical prompts twice.
    items = [(qid, q, ev)]
    a1 = answer_batch(items)[0]
    # Remove this prompt from in-memory cache only, then regenerate deterministically.
    key = answer_cache_key(q, ev)
    saved = answer_cache.pop(key, None)
    a2 = answer_batch(items)[0]
    assert a1 == a2, (qid, a1, a2)

print("DETERMINISTIC ANSWER SMOKE — PASS")

In [ ]:
# Cell 11 — Reconstruct frozen centroid trajectories and save terminal candidate rows
TERMINAL_PATH = OUT / "v036_terminal_candidates.parquet"
ROUND0_PATH = OUT / "v036_round0_candidates.parquet"

if TERMINAL_PATH.is_file() and ROUND0_PATH.is_file():
    terminal = pd.read_parquet(TERMINAL_PATH)
    round0 = pd.read_parquet(ROUND0_PATH)
    print("Reusing candidate artifacts.")
else:
    # Round-0 retrieval is policy-independent.
    _, I_rep0 = batched_search(pq_cpu, Q0, REP_NPROBE)
    _, I_high0 = batched_search(sq_cpu, Q0, SEARCH_HIGH_NPROBE)
    _, I_search0 = batched_search(sq_cpu, Q0, SEARCH_LOW_NPROBE)

    round0_rows = []
    for i, qid in enumerate(MAIN_IDS):
        for branch, I in [("rep_low",I_rep0),("high",I_high0),("search_low",I_search0)]:
            round0_rows.append({
                "query_id":qid,
                "branch":branch,
                "rows_json":json.dumps(list(map(int, I[i,:EVIDENCE_K]))),
            })
    round0 = pd.DataFrame(round0_rows)
    round0.to_parquet(ROUND0_PATH, index=False)

    term_rows = []
    for pi, policy in enumerate(POLICIES):
        print(f"[{pi+1}/{len(POLICIES)}] {policy['name']}")
        states = {
            "rep_low": Q0.copy(),
            "high": Q0.copy(),
            "search_low": Q0.copy(),
        }

        for t in range(H+1):
            D_rep, I_rep = batched_search(pq_cpu, states["rep_low"], REP_NPROBE)
            D_high, I_high = batched_search(sq_cpu, states["high"], SEARCH_HIGH_NPROBE)
            D_search, I_search = batched_search(sq_cpu, states["search_low"], SEARCH_LOW_NPROBE)

            if t == H:
                for i, qid in enumerate(MAIN_IDS):
                    for branch, I in [
                        ("rep_low",I_rep), ("high",I_high), ("search_low",I_search)
                    ]:
                        term_rows.append({
                            "query_id":qid,
                            "policy":policy["name"],
                            "family":policy["family"],
                            "alpha":policy["alpha"],
                            "branch":branch,
                            "rows_json":json.dumps(list(map(int, I[i,:EVIDENCE_K]))),
                        })
                break

            F_rep = feedback(I_rep, D_rep, policy)
            F_high = feedback(I_high, D_high, policy)
            F_search = feedback(I_search, D_search, policy)

            states["rep_low"] = anchored_update(Q0, F_rep, policy["alpha"])
            states["high"] = anchored_update(Q0, F_high, policy["alpha"])
            states["search_low"] = anchored_update(Q0, F_search, policy["alpha"])

        pd.DataFrame(term_rows).to_parquet(
            OUT / "v036_terminal_candidates_partial.parquet", index=False
        )
        gc.collect()

    terminal = pd.DataFrame(term_rows)
    terminal.to_parquet(TERMINAL_PATH, index=False)

assert len(round0) == len(MAIN_IDS)*3
assert len(terminal) == len(MAIN_IDS)*len(POLICIES)*3
print("Candidate artifacts ready:", len(round0), len(terminal))

In [ ]:
# Cell 12 — Generate and score one-shot answers
ROUND0_ANS_PATH = OUT / "v036_round0_answers.parquet"

if ROUND0_ANS_PATH.is_file():
    round0_ans = pd.read_parquet(ROUND0_ANS_PATH)
else:
    items, meta = [], []
    for r in round0.itertuples(index=False):
        rows = json.loads(r.rows_json)
        ev = evidence_for_rows(rows)
        items.append((r.query_id, QUERY_TEXT[r.query_id], ev))
        meta.append((r.query_id, r.branch))
    preds = []
    for s in tqdm(range(0, len(items), 64), desc="One-shot answer batches"):
        preds.extend(answer_batch(items[s:s+64]))

    rows_out = []
    for (qid, branch), pred in zip(meta, preds):
        golds = GOLD[qid]
        rows_out.append({
            "query_id":qid,
            "branch":branch,
            "prediction":pred,
            "f1":token_f1(pred, golds),
            "em":exact_match(pred, golds),
        })
    round0_ans = pd.DataFrame(rows_out)
    round0_ans.to_parquet(ROUND0_ANS_PATH, index=False)

display(round0_ans.groupby("branch")[["f1","em"]].mean())

In [ ]:
# Cell 13 — Generate and score terminal answers
TERMINAL_ANS_PATH = OUT / "v036_terminal_answers.parquet"

if TERMINAL_ANS_PATH.is_file():
    terminal_ans = pd.read_parquet(TERMINAL_ANS_PATH)
else:
    items, meta = [], []
    for r in terminal.itertuples(index=False):
        rows = json.loads(r.rows_json)
        ev = evidence_for_rows(rows)
        items.append((r.query_id, QUERY_TEXT[r.query_id], ev))
        meta.append((r.query_id, r.policy, r.family, r.alpha, r.branch))

    preds = []
    for s in tqdm(range(0, len(items), 64), desc="Terminal answer batches"):
        preds.extend(answer_batch(items[s:s+64]))

    rows_out = []
    for (qid, policy, family, alpha, branch), pred in zip(meta, preds):
        golds = GOLD[qid]
        rows_out.append({
            "query_id":qid,
            "policy":policy,
            "family":family,
            "alpha":alpha,
            "branch":branch,
            "prediction":pred,
            "f1":token_f1(pred, golds),
            "em":exact_match(pred, golds),
        })
    terminal_ans = pd.DataFrame(rows_out)
    terminal_ans.to_parquet(TERMINAL_ANS_PATH, index=False)

display(terminal_ans.groupby("branch")[["f1","em"]].mean())

In [ ]:
# Cell 14 — Frozen primary and secondary inference
def paired_bootstrap(v, reps=BOOTSTRAP_REPS, seed=SEED):
    v = np.asarray(v, dtype=np.float64)
    assert np.isfinite(v).all()
    rng = np.random.default_rng(seed)
    n = len(v)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 500
    pos = 0
    while pos < reps:
        m = min(chunk, reps-pos)
        idx = rng.integers(0, n, size=(m,n))
        boots[pos:pos+m] = v[idx].mean(axis=1)
        pos += m
    return {
        "n": int(n),
        "mean": float(v.mean()),
        "ci95": [
            float(np.quantile(boots, 0.025)),
            float(np.quantile(boots, 0.975)),
        ],
    }

# One-shot branch table: query x branch
r0 = round0_ans.pivot(index="query_id", columns="branch", values=["f1","em"])

# Terminal: average frozen policies within query first.
term_q = (
    terminal_ans
    .groupby(["query_id","branch"], as_index=False)[["f1","em"]]
    .mean()
)
th = term_q.pivot(index="query_id", columns="branch", values=["f1","em"])

common = sorted(set(r0.index) & set(th.index))
assert len(common) == len(MAIN_IDS) == 500

D0_f1 = r0.loc[common, ("f1","search_low")].to_numpy() - r0.loc[common, ("f1","rep_low")].to_numpy()
DH_f1 = th.loc[common, ("f1","search_low")].to_numpy() - th.loc[common, ("f1","rep_low")].to_numpy()
delta_f1 = DH_f1 - D0_f1

D0_em = r0.loc[common, ("em","search_low")].to_numpy() - r0.loc[common, ("em","rep_low")].to_numpy()
DH_em = th.loc[common, ("em","search_low")].to_numpy() - th.loc[common, ("em","rep_low")].to_numpy()
delta_em = DH_em - D0_em

primary = paired_bootstrap(delta_f1, seed=SEED+1)
lo, hi = primary["ci95"]
primary["classification"] = (
    "POSITIVE_FEEDBACK_ADDED_QA_CONSEQUENCE" if lo > 0 else
    "REVERSED_FEEDBACK_ADDED_QA_CONSEQUENCE" if hi < 0 else
    "UNRESOLVED"
)

secondary = {
    "terminal_F1_search_minus_rep": paired_bootstrap(DH_f1, seed=SEED+2),
    "oneshot_F1_search_minus_rep": paired_bootstrap(D0_f1, seed=SEED+3),
    "feedback_added_EM_contrast": paired_bootstrap(delta_em, seed=SEED+4),
    "terminal_EM_search_minus_rep": paired_bootstrap(DH_em, seed=SEED+5),
    "oneshot_EM_search_minus_rep": paired_bootstrap(D0_em, seed=SEED+6),
    "terminal_high_minus_rep_F1": paired_bootstrap(
        th.loc[common,("f1","high")].to_numpy() - th.loc[common,("f1","rep_low")].to_numpy(),
        seed=SEED+7
    ),
    "terminal_high_minus_search_F1": paired_bootstrap(
        th.loc[common,("f1","high")].to_numpy() - th.loc[common,("f1","search_low")].to_numpy(),
        seed=SEED+8
    ),
}

qout = pd.DataFrame({
    "query_id":common,
    "D0_f1_search_minus_rep":D0_f1,
    "DH_f1_search_minus_rep":DH_f1,
    "feedback_added_f1_contrast":delta_f1,
    "D0_em_search_minus_rep":D0_em,
    "DH_em_search_minus_rep":DH_em,
    "feedback_added_em_contrast":delta_em,
})
qout.to_csv(OUT / "v036_query_level_answer_endpoints.csv", index=False)

result = {
    "study_id":"ARC-v0.36.3",
    "protocol_sha256":PROTOCOL_SHA,
    "primary":primary,
    "secondary":secondary,
    "descriptive": {
        "round0_branch_mean": round0_ans.groupby("branch")[["f1","em"]].mean().to_dict(),
        "terminal_branch_mean": terminal_ans.groupby("branch")[["f1","em"]].mean().to_dict(),
        "fraction_queries_primary_positive": float(np.mean(delta_f1 > 0)),
        "fraction_queries_primary_negative": float(np.mean(delta_f1 < 0)),
        "fraction_queries_primary_tied": float(np.mean(delta_f1 == 0)),
    },
    "retuned_after_answer_outcomes": False,
}
(OUT / "v036_primary_gate.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

print(json.dumps(primary, indent=2))

In [ ]:
# Cell 15 — Conservative manuscript-facing interpretation and artifact hashes
p = result["primary"]
lo, hi = p["ci95"]

if p["classification"] == "POSITIVE_FEEDBACK_ADDED_QA_CONSEQUENCE":
    wording = (
        "In a frozen post-primary NQ-GTE downstream audit, the representation-versus-search "
        "trajectory difference propagates to answer quality beyond the one-shot answer difference: "
        f"Delta_QA={p['mean']:+.4f} token-F1, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "This is evidence for one fixed answer model, one 500-query source subset, anchored H=4, "
        "and the frozen eight-policy set; it is not a universal RAG answer-quality law."
    )
elif p["classification"] == "REVERSED_FEEDBACK_ADDED_QA_CONSEQUENCE":
    wording = (
        "The frozen downstream QA audit reverses the expected feedback-added answer-quality ordering: "
        f"Delta_QA={p['mean']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "Report this as an answer-level boundary rather than retuning the endpoint."
    )
else:
    wording = (
        "The frozen downstream QA audit is unresolved for feedback-added answer-quality ordering: "
        f"Delta_QA={p['mean']:+.4f}, 95% CI [{lo:+.4f}, {hi:+.4f}]. "
        "The retrieval-trajectory findings therefore should not be promoted to a signed downstream "
        "answer-quality claim under this answerer."
    )

report = {
    "study_id":"ARC-v0.36.3",
    "status":"COMPLETE",
    "primary":p,
    "suggested_manuscript_wording":wording,
    "claim_guardrail": (
        "Post-primary new answer endpoint. Source retrieval outcomes pre-existed. "
        "Do not call this a pristine independent confirmation."
    ),
}
(OUT / "v036_final_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

artifacts = []
for path in sorted(OUT.iterdir()):
    if path.is_file() and path.name != "V036_ARTIFACT_SHA256.csv":
        artifacts.append({
            "file":path.name,
            "bytes":path.stat().st_size,
            "sha256":sha256_file(path),
        })
pd.DataFrame(artifacts).to_csv(OUT / "V036_ARTIFACT_SHA256.csv", index=False)

print(wording)
print("\nArtifacts:", OUT)

## Reporting guardrail

ARC-v0.36 can strengthen the paper only at the level supported by its result.

- **Positive CI:** downstream answer quality provides one post-primary consequence audit consistent with the retrieval mechanism story.
- **CI crosses zero:** report answer-level transfer as unresolved; do not hide it and do not change the answer model or primary.
- **Negative CI:** retain the reversal as a downstream boundary.

Regardless of outcome, do not describe ARC-v0.36 as an independent pristine confirmation: the source retrieval experiments and the 500-query membership were already known before this answer endpoint was introduced.